# Module 13 — Authentication + User & Organization Management

This notebook evaluates the PriceMind AI authentication & multi-tenant user architecture.

**Architecture:**
```
User → React UI (Login/Signup) → FastAPI (/api/v1/auth) → bcrypt / JWT → PostgreSQL (Users/Orgs)
```


In [ ]:
import sys
from pathlib import Path

# Add project root and backend to path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'backend'))
print('Project root:', ROOT)


## 1. Password Hashing & Verification (Bcrypt)


In [ ]:
from app.core.security import hash_password, verify_password

password = 'EnterprisePassword2026!'
hashed = hash_password(password)

print('Plaintext:', password)
print('Bcrypt Hash:', hashed)
print('Verify correct password:', verify_password(password, hashed))
print('Verify wrong password:', verify_password('WrongPassword', hashed))


## 2. JWT Access Token Creation & Validation


In [ ]:
from datetime import timedelta
from app.core.security import create_access_token, decode_access_token

claims = {
    'sub': 'user-8921-prod',
    'email': 'analyst@pricemind.ai',
    'organization_id': 'org-enterprise-01',
    'role': 'analyst',
}

token = create_access_token(claims, expires_delta=timedelta(hours=24))
print('Generated JWT Token:')
print(token[:40] + '...' + token[-20:])

decoded = decode_access_token(token)
print('\nDecoded Claims:')
for k, v in decoded.items():
    print(f'  {k}: {v}')


## 3. End-to-End Auth API Flow (TestClient)


In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

# 1. Register a new user + org
reg_payload = {
    'full_name': 'Dr. Elena Rostova',
    'email': 'elena.rostova@industrial-pricing.org',
    'organization_name': 'Industrial Pricing Dynamics',
    'password': 'SecureEnterprisePassword2026!',
    'confirm_password': 'SecureEnterprisePassword2026!',
}

reg_res = client.post('/api/v1/auth/register', json=reg_payload)
print('Register status:', reg_res.status_code)
if reg_res.status_code == 201:
    token_data = reg_res.json()
    token = token_data['access_token']
    print('User registered:', token_data['user']['email'])
    print('Organization:', token_data['user']['organization_name'])
else:
    # If already registered, login
    login_res = client.post('/api/v1/auth/login', json={
        'email': reg_payload['email'],
        'password': reg_payload['password']
    })
    token_data = login_res.json()
    token = token_data['access_token']
    print('User logged in:', token_data['user']['email'])

# 2. Call /auth/me with Bearer token
me_res = client.get('/api/v1/auth/me', headers={'Authorization': f'Bearer {token}'})
print('\n/auth/me Profile:')
print(json.dumps(me_res.json(), indent=2))


## 4. Protected Route Verification


In [ ]:
# Unauthenticated query should return 401
unauth_res = client.post('/api/v1/agent/query', json={'message': 'What is the margin floor?'})
print('Unauthenticated /agent/query status:', unauth_res.status_code)  # 401

# Authenticated query with valid Bearer token should succeed
auth_res = client.post(
    '/api/v1/agent/query',
    json={'message': 'What is the margin floor policy?'},
    headers={'Authorization': f'Bearer {token}'}
)
print('Authenticated /agent/query status:', auth_res.status_code)  # 200
if auth_res.status_code == 200:
    print('Answer preview:', auth_res.json()['answer'][:200])
